In [1]:
# Imports unificados e setup
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
from bizdays import Calendar
import plotly.express as px
import plotly.graph_objects as go

from analise_liquidez.dados import DEBENTURES_SCHEMA, carregar_dados_resiliente, ajustar_tipos
from analise_liquidez.regras import expandir_dias_uteis, filtrar_periodo_recente, calcular_medias_e_volumes, calcular_liquidez_carteira
from analise_liquidez.graficos import plot_titulo_plotly


In [2]:
# Carregamento resiliente dos dados de debêntures
url = "https://raw.githubusercontent.com/royopa/PulseFlat/main/data/debentures_mercado_secundario_precos_negociacao.csv.gz"
local_path = "../../PulseFlat/data/debentures_mercado_secundario_precos_negociacao.csv.gz"
if not os.path.exists(local_path):
    local_path = "../PulseFlat/data/debentures_mercado_secundario_precos_negociacao.csv.gz"

df_raw = carregar_dados_resiliente(local_path, url)

# Limpeza de metadados desnecessários
for col in ['arquivo_origem', 'data_captura', 'conjunto', 'registro_hash']:
    if col in df_raw.columns:
        del df_raw[col]

# Ajuste de tipos de dados baseado no schema unificado
df_adjusted = ajustar_tipos(df_raw, DEBENTURES_SCHEMA)

display(df_adjusted.head())


Carregando dados de: ../../PulseFlat/data/debentures_mercado_secundario_precos_negociacao.csv.gz


,data_referencia,emissor,codigo_ativo,isin,quantidade,numero_de_negocios,pu_minimo,pu_medio,pu_maximo,pu_da_curva
0,2020-01-31,AGUAS GUARIROBA S/A,AGGU13,BRGRRBDBS026,200,2,1.027686e+10,1.027839e+10,1.027992e+10,10041.0
1,2020-01-31,CONCESSIONARIA DO AEROPORTO INTERNACIONAL DE G...,AGRU11,BRAGRUDBS006,2,1,1.213183e+09,1.213183e+09,1.213183e+09,9851.0
2,2020-01-31,CONCESSIONARIA DO AEROPORTO INTERNACIONAL DE G...,AGRU12,BRAGRUDBS048,30,3,1.153664e+09,1.191964e+09,1.268564e+09,9728.0
3,2020-01-31,CONCESSIONARIA DO AEROPORTO INTERNACIONAL DE G...,AGRU31,BRAGRUDBS022,9,4,1.192627e+09,1.232432e+09,1.264616e+09,10399.0
4,2020-01-31,ALGAR TELECOM S.A.,ALGA26,BRALGTDBS050,303,3,1.316560e+05,1.326708e+09,1.326776e+08,11269.0


In [3]:
# Carrega calendário de negócios ANBIMA e expande o histórico por dia útil
cal = Calendar.load('ANBIMA')
min_date = df_adjusted['data_referencia'].min()
max_date = df_adjusted['data_referencia'].max()

print(f"Data mínima: {min_date.date()} | Data máxima: {max_date.date()}")

business_days = cal.seq(min_date, max_date)
business_days_ts = pd.to_datetime(business_days)

df_expanded = expandir_dias_uteis(df_adjusted, business_days_ts, DEBENTURES_SCHEMA)

print("DataFrame expandido (primeiras linhas):")
display(df_expanded.head())


Data mínima: 2020-01-02 | Data máxima: 2026-07-30


DataFrame expandido (primeiras linhas):


,data_referencia,emissor,codigo_ativo,isin,quantidade,numero_de_negocios,pu_minimo,pu_medio,pu_maximo,pu_da_curva
0,2020-01-02,ORIGEM ENERGIA S.A.,ORIG11,BR0HJ5DBS000,4147,83,100670257.0,1.034394e+09,1.133112e+09,10269.0
1,2020-01-03,ORIGEM ENERGIA S.A.,ORIG11,BR0HJ5DBS000,4147,83,100670257.0,1.034394e+09,1.133112e+09,10269.0
2,2020-01-06,ORIGEM ENERGIA S.A.,ORIG11,BR0HJ5DBS000,4147,83,100670257.0,1.034394e+09,1.133112e+09,10269.0
3,2020-01-07,ORIGEM ENERGIA S.A.,ORIG11,BR0HJ5DBS000,4147,83,100670257.0,1.034394e+09,1.133112e+09,10269.0
4,2020-01-08,ORIGEM ENERGIA S.A.,ORIG11,BR0HJ5DBS000,4147,83,100670257.0,1.034394e+09,1.133112e+09,10269.0


In [4]:
# Filtro para os últimos 3 meses usando a data de referência padrão de debêntures (julho de 2026)
data_referencia = pd.to_datetime('2026-07-30')
print(f"Data de Referência de Análise: {data_referencia.date()}")

df_analise_recent = filtrar_periodo_recente(df_expanded, data_referencia, months_ago=3, schema=DEBENTURES_SCHEMA)
df_analise_recent = ajustar_tipos(df_analise_recent, DEBENTURES_SCHEMA)

display(df_analise_recent.head())


Data de Referência de Análise: 2026-07-30
DataFrame filtrado para o período: 2026-04-30 a 2026-07-30


,data_referencia,emissor,codigo_ativo,isin,quantidade,numero_de_negocios,pu_minimo,pu_medio,pu_maximo,pu_da_curva
1587,2026-04-30,ORIGEM ENERGIA S.A.,ORIG11,BR0HJ5DBS000,669,14,8.610216e+09,9.643953e+09,9.983454e+09,83510.0
1588,2026-05-04,ORIGEM ENERGIA S.A.,ORIG11,BR0HJ5DBS000,305,14,9.880741e+09,1.003358e+10,1.092465e+10,86820.0
1589,2026-05-05,ORIGEM ENERGIA S.A.,ORIG11,BR0HJ5DBS000,968,16,8.715006e+09,9.665493e+09,1.065266e+10,83580.0
1590,2026-05-06,ORIGEM ENERGIA S.A.,ORIG11,BR0HJ5DBS000,592,16,9.371021e+09,9.819508e+09,1.073909e+10,84860.0
1591,2026-05-07,ORIGEM ENERGIA S.A.,ORIG11,BR0HJ5DBS000,366,10,8.748425e+09,1.012916e+10,1.071709e+09,87480.0


In [5]:
# Identificação das debêntures mais negociadas por número de negócios no período recente
top_5_negociados_recentes = df_analise_recent.groupby('isin')['numero_de_negocios'].sum().nlargest(5)
top_5_isins = top_5_negociados_recentes.index.tolist()

top_5_info = df_analise_recent[df_analise_recent['isin'].isin(top_5_isins)][['isin', 'codigo_ativo']].drop_duplicates()
top_5_info = top_5_info.set_index('isin').loc[top_5_isins].reset_index()
top_5_info = top_5_info.merge(top_5_negociados_recentes.rename('total_negocios'), on='isin')

print("Top 5 Debêntures Mais Negociadas:")
display(top_5_info)


Top 5 Debêntures Mais Negociadas:


,isin,codigo_ativo,total_negocios
0,BRAMERDBS008,BTOW15,61952
1,BRAMERDBS099,LAMEA7,38976
2,BROLIPDBS006,OLIP11,19392
3,BRSEERDBS049,SEER13,13440
4,BRCEALDBS010,CEAL11,10112


In [6]:
# Gráfico interativo com duas médias móveis (7 e 30 dias) para a debênture mais negociada
isin_mais_negociado = top_5_isins[0]
codigo_ativo_mais_negociado = top_5_info[top_5_info['isin'] == isin_mais_negociado]['codigo_ativo'].iloc[0]

fig_plotly = plot_titulo_plotly(
    df_data=df_analise_recent,
    isin_to_plot=isin_mais_negociado,
    codigo_ativo_to_plot=codigo_ativo_mais_negociado,
    schema=DEBENTURES_SCHEMA,
    y_column='numero_de_negocios',
    title_suffix="(Últimos 3 Meses com Duas Médias Móveis)",
    window_size=7,
    window_size_secondary=30
)
fig_plotly.show()


In [7]:
# Cálculo das médias móveis de quantidade de 21 dias úteis e volumes de liquidez
df_analise = calcular_medias_e_volumes(df_expanded, DEBENTURES_SCHEMA)
display(df_analise.sort_values(by=['data_referencia', 'quantidade_media_21_dias'], ascending=[False, False]).head())


,data_referencia,emissor,codigo_ativo,isin,quantidade,numero_de_negocios,pu_minimo,pu_medio,pu_maximo,pu_da_curva,quantidade_media_21_dias,valor_total_sem_desconto,volume_liquido_1dia_calculado
1650,2026-07-30,QUEIROZ GALVAO S/A,QGSA26,BRQGSADBS044,394425510,2,12461.0,12461.0,12461.0,100.0,394425510,4914936280110.0,4914936280110.0
1650,2026-07-30,MARLIN NAVEGACAO SA,MLNV12,BRMLNVDBS016,368592732,2,16054.0,16054.0,16054.0,9019.0,368592732,5917387719528.0,5917387719528.0
1650,2026-07-30,ERBE INCORPORADORA S.A.,BISA16,BRBISADBS0A8,350000000,1,615154.0,615154.0,615154.0,NaN,350000000,215303900000000.0,215303900000000.0
1650,2026-07-30,TRAVESSIA SECURITIZADORA DE CREDITOS FINANCEIR...,PCHSE0,BRPCHSDBS0R5,325000000,1,2354464.0,2354464.0,2354464.0,NaN,325000000,765200800000000.0,765200800000000.0
1650,2026-07-30,SC2 MARANHAO LOCACAO DE CENTROS COMERCIAIS S/A,SCMR41,BRSCMRDBS038,293958396,1,15308.0,15308.0,15308.0,8319.0,293958396,4499915125968.0,4499915125968.0


In [8]:
# Gráfico de barras de top 10 ISINs por volume médio diário acumulado
top_10_media_21_dias = df_analise.groupby('isin')['quantidade_media_21_dias'].sum().nlargest(10).reset_index()
top_10_info_media = pd.merge(
    top_10_media_21_dias,
    df_analise[['isin', 'codigo_ativo']].drop_duplicates(),
    on='isin',
    how='left'
)
top_10_info_media['isin_codigo'] = top_10_info_media['isin'] + ' (' + top_10_info_media['codigo_ativo'].astype(str) + ')'

fig_bar = px.bar(
    top_10_info_media,
    x='isin_codigo',
    y='quantidade_media_21_dias',
    title='Top 10 Debêntures por Quantidade Média de 21 Dias',
    labels={'isin_codigo': 'ISIN (Código Ativo)', 'quantidade_media_21_dias': 'Soma da Quantidade Média de 21 Dias'}
)
fig_bar.update_layout(xaxis={'categoryorder':'total descending'})
fig_bar.show()


In [9]:
# Análise de Liquidez de Carteira do Fundo
parametros_calculo = {
    "data_referencia": data_referencia,
    "prazo_de_cotizacao": 1,
    "carteira_de_debentures": [
        ["BRMGPRDBS068", "MTRJ19", 5000],
        ["BRPETRDBS0C2", "PETR27", 12000],
        ["BRCRBDDBS025", "CBAN12", 8000]
    ]
}

df_analise_liquidez_carteira = calcular_liquidez_carteira(
    parametros=parametros_calculo,
    df_dados_liquidez=df_analise,
    schema=DEBENTURES_SCHEMA
)

print("\n--- Resultado da Análise de Liquidez da Carteira ---")
pd.options.display.float_format = '{:,.2f}'.format
display(df_analise_liquidez_carteira.head())



--- Análise de Liquidez para a Carteira do Fundo na data 2026-07-30 ---
  Ativo: MTRJ19 (ISIN: BRMGPRDBS068) - Dados de liquidez encontrados.


  Ativo: PETR27 (ISIN: BRPETRDBS0C2) - Dados de liquidez encontrados.
  Ativo: CBAN12 (ISIN: BRCRBDDBS025) - Dados de liquidez encontrados.

--- Resultado da Análise de Liquidez da Carteira ---


,isin,codigo_ativo,quantidade_fundo,data_referencia,pu_medio,liquidez_media_21_dias,volume_liquido_1dia_calculado,valor_total_alocado,valor_total_prazo_cotizacao,valor_total_liquido,percentual_liquido_alocado
0,BRMGPRDBS068,MTRJ19,5000,2026-07-30,"1,276,158,061.00",1840,"2,348,130,832,240.00","6,380,790,305,000.00","2,348,130,832,240.00","2,348,130,832,240.00",36.80
1,BRPETRDBS0C2,PETR27,12000,2026-07-30,"1,180,185,279.00",986,"1,163,662,685,094.00","14,162,223,348,000.00","1,163,662,685,094.00","1,163,662,685,094.00",8.22
2,BRCRBDDBS025,CBAN12,8000,2026-07-30,"1,402,344,841.00",830,"1,163,946,218,030.00","11,218,758,728,000.00","1,163,946,218,030.00","1,163,946,218,030.00",10.38
